# Lesson 3.1 — Tool Use and Function Calling Basics

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Define a tool with name, description, and input_schema
- Run the full tool-use loop: request, tool_use, execute, tool_result
- Read stop_reason == 'tool_use' and dispatch to your code
- Return tool results so Claude can produce a final answer

**You will produce:** A working single-tool loop (a calculator) that runs fully offline-safe.

## Phase 3 Overview

Phase 3 covers **Domain 8 - Tools and MCPs (10.6%)**: implementing tools and function calling, developing MCP servers, and choosing among built-in tools, custom tools, Skills, and MCPs. You will build tools and a real MCP server stub.

By the end of this phase, you will have built:

- A single-tool and multi-tool agentic loop
- Well-written tool descriptions and schemas
- Client-side vs server-side tool patterns with approval gating
- An MCP server stub (resources, tools, prompts) over stdio
- A decision guide across built-in tools, custom tools, Skills, and MCPs

### Prerequisites

- Completed Phases 1-2
- `pip install anthropic`
- Optional: `pip install mcp` for the MCP lessons
- Comfort with JSON Schema basics

## Concepts

**Tool use** (function calling) lets Claude request that *your* code run a function. A tool is a dict with `name`, `description`, and an `input_schema` (JSON Schema). The loop:

1. You send `messages` plus `tools=[...]`.
2. Claude may reply with `stop_reason == 'tool_use'` and a `tool_use` block (name + input).
3. **You** execute the function and send a `tool_result` block (referencing the `tool_use_id`) back as a new user turn.
4. Claude uses the result to produce the final answer.

Claude never runs code itself - it only *asks*. Your harness dispatches the call. This separation is central to Domain 8 and to safe design (Phase 5).

> **Reference:** [Anthropic - Tool use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Define a calculator tool

Name it clearly and describe exactly when to use it.

In [ ]:
CALC_TOOL = {
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression. Use for any math.",
    "input_schema": {
        "type": "object",
        "properties": {"expression": {"type": "string",
            "description": "e.g. '2 * (3 + 4)'"}},
        "required": ["expression"],
    },
}

def run_calculator(expression):
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "error: invalid characters"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"error: {e}"

print(run_calculator("2 * (3 + 4)"))

## Code-Along: Step 2 — The tool-use loop

Send the tool, detect tool_use, execute, and return a tool_result.

In [ ]:
def ask_with_tool(question):
    if not HAVE_API:
        # Offline: simulate the loop deterministically
        return f"[MOCK] 2*(3+4) = {run_calculator('2*(3+4)')}"
    messages = [{"role": "user", "content": question}]
    msg = client.messages.create(model=MODEL_SONNET, max_tokens=500,
        tools=[CALC_TOOL], messages=messages)
    while msg.stop_reason == "tool_use":
        messages.append({"role": "assistant", "content": msg.content})
        results = []
        for block in msg.content:
            if block.type == "tool_use" and block.name == "calculator":
                out = run_calculator(block.input["expression"])
                results.append({"type": "tool_result",
                    "tool_use_id": block.id, "content": out})
        messages.append({"role": "user", "content": results})
        msg = client.messages.create(model=MODEL_SONNET, max_tokens=500,
            tools=[CALC_TOOL], messages=messages)
    return msg.content[0].text

print(ask_with_tool("What is 2 times (3 plus 4)?"))

## Exercise

Add a second tool `get_time` (no inputs) that returns the current UTC time, and extend the dispatch in the loop to handle whichever tool Claude calls.

In [ ]:
import datetime
TIME_TOOL = {
    "name": "get_time", "description": "Return current UTC time.",
    "input_schema": {"type": "object", "properties": {}},
}
def run_get_time():
    # TODO: return datetime.datetime.utcnow().isoformat()
    pass
# TODO: add dispatch for 'get_time' alongside 'calculator' in the loop

## Key Takeaways

- A tool is name + description + input_schema (JSON Schema).
- Claude only requests tools; your harness executes them and returns tool_result.
- Loop while stop_reason == 'tool_use', echoing assistant content then user tool_result.
- Reference the tool_use_id in each tool_result so Claude matches request to response.

---

**Next:** [Lesson 3.2 — Writing Tool Descriptions and Building Tool Sets](./lesson-3.2-tool-descriptions-and-sets.ipynb)